# 01 Full Pipeline Visual Analysis

Dieses Notebook macht die EEG-Pipeline fuer IDUN/Guardian CSV-Dateien visuell und analytisch nachvollziehbar. Der Fokus liegt auf sauberer Signalverarbeitung, Artefaktmarkierung, Feature Engineering und vorsichtiger Interpretation.

Wichtig: Der Cognitive Load Proxy Score ist ein baseline-relativer EEG-Index. Ohne externen Referenzscore wie NASA-TLX wird kein echter Cognitive Overload behauptet.

## 1. Projekt-Setup

Imports, Pfade und zentrale Konfiguration. Alle Notebook-Plots werden als PNG unter `outputs/plots/notebook/` gespeichert.

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, iirnotch, sosfiltfilt, welch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib.util

PIPELINE_PATH = PROJECT_ROOT / "01_ordered_eeg_pipeline.py"
spec = importlib.util.spec_from_file_location("ordered_eeg_pipeline", PIPELINE_PATH)
ordered = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ordered)

CONFIG = ordered.CONFIG
FREQUENCY_BANDS = ordered.FREQUENCY_BANDS
EPS = ordered.EPS
USE_EXTERNAL_SCORE = ordered.USE_EXTERNAL_SCORE
EXTERNAL_SCORE_COLUMN = ordered.EXTERNAL_SCORE_COLUMN
EXTERNAL_SCORE_TYPE = ordered.EXTERNAL_SCORE_TYPE
HIGH_WORKLOAD_THRESHOLD = ordered.HIGH_WORKLOAD_THRESHOLD
_validate_columns = ordered._validate_columns
_check_sampling_rate = ordered._check_sampling_rate
_create_windows = ordered._create_windows
_compute_artifact_metrics = ordered._compute_artifact_metrics
_artifact_thresholds = ordered._artifact_thresholds
_bandpower = ordered._bandpower
_spectral_entropy = ordered._spectral_entropy
_baseline_normalize = ordered._baseline_normalize
_assign_load_states = ordered._assign_load_states
_add_eeg_deviation_index = ordered._add_eeg_deviation_index
_attach_and_validate_external_score = ordered._attach_and_validate_external_score
_compute_proxy_score = ordered._compute_proxy_score

PREPROCESSING_CONFIG = CONFIG["preprocessing"]
WINDOWING_CONFIG = CONFIG["windowing"]
PSD_CONFIG = CONFIG["psd"]
BASELINE_CONFIG = CONFIG["baseline"]
PROXY_CONFIG = CONFIG["cognitive_load_proxy"]
PLOT_CONFIG = CONFIG["plots"]
BANDPASS_LOW_HZ = float(PREPROCESSING_CONFIG["bandpass_low_hz"])
BANDPASS_HIGH_HZ = float(PREPROCESSING_CONFIG["bandpass_high_hz"])
BANDPASS_ORDER = int(PREPROCESSING_CONFIG["bandpass_order"])
NOTCH_FREQ_HZ = float(PREPROCESSING_CONFIG["notch_freq_hz"])
NOTCH_QUALITY_FACTOR = float(PREPROCESSING_CONFIG["notch_quality_factor"])
WINDOW_SECONDS = float(WINDOWING_CONFIG["window_seconds"])
WINDOW_OVERLAP = float(WINDOWING_CONFIG["overlap"])
PSD_NPERSEG_MAX = int(PSD_CONFIG["nperseg_max"])
PSD_FULL_SIGNAL_NPERSEG_MAX = int(PSD_CONFIG["full_signal_nperseg_max"])
TOTAL_POWER_BAND = (float(PSD_CONFIG["total_power_low_hz"]), float(PSD_CONFIG["total_power_high_hz"]))
BASELINE_MINUTES = float(BASELINE_CONFIG["baseline_minutes"])
ROLLING_WINDOWS = int(PROXY_CONFIG["rolling_windows"])
DETAIL_SECONDS = float(PLOT_CONFIG["detail_seconds"])

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PLOT_DIR = PROJECT_ROOT / PLOT_CONFIG["notebook_output_dir"]
EXPORT_DIR = PROJECT_ROOT / "outputs" / "notebook"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("default")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

def savefig(name):
    path = PLOT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=int(PLOT_CONFIG["dpi"]), bbox_inches="tight")
    print(f"Gespeichert: {path}")
    plt.show()

print("Projektordner:", PROJECT_ROOT)
print("Raw-Daten:", RAW_DIR)
print("Notebook-Plots:", PLOT_DIR)

## 2. CSV-Datei aus data/raw auswählen

Setze `CSV_NAME` auf einen konkreten Dateinamen oder lasse `None`, um automatisch die erste CSV aus `data/raw` zu verwenden.

In [ ]:
CSV_NAME = None  # Beispiel: "eeg_Work_PC_Morning.csv"

csv_files = sorted(RAW_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"Keine CSV-Dateien in {RAW_DIR} gefunden.")

if CSV_NAME is None:
    csv_path = csv_files[0]
else:
    csv_path = RAW_DIR / CSV_NAME

print("Verfügbare CSV-Dateien:")
for file in csv_files:
    print("-", file.name)
print("\nAusgewählte Datei:", csv_path)

## 3. Rohdaten laden

Die CSV wird geladen, die Zeitspalte automatisch erkannt und `ch1` numerisch konvertiert. Falls ein externer Workload-Score vorhanden ist, wird er nur als Referenz vorbereitet.

In [ ]:
raw_df = pd.read_csv(csv_path)
timestamp_col = _validate_columns(raw_df)

df = raw_df.copy()
df[timestamp_col] = pd.to_numeric(df[timestamp_col], errors="coerce")
df["ch1"] = pd.to_numeric(df["ch1"], errors="coerce")
external_score_available = bool(USE_EXTERNAL_SCORE and EXTERNAL_SCORE_COLUMN in df.columns)
if external_score_available:
    df[EXTERNAL_SCORE_COLUMN] = pd.to_numeric(df[EXTERNAL_SCORE_COLUMN], errors="coerce")
    print("Externer Score erkannt:", EXTERNAL_SCORE_COLUMN)
else:
    print("Hinweis: Kein externer Workload-Score vorhanden.")

display(raw_df.head())
print("Shape roh:", raw_df.shape)
print("Zeitspalte:", timestamp_col)

## 4. Datenvalidierung anzeigen

Fehlende Werte, Duplikate und Sortierung werden geprüft. Danach liegt ein sauber sortiertes Rohsignal vor.

In [ ]:
rows_before = len(df)
missing_before = df[[timestamp_col, "ch1"]].isna().sum()
df = df.dropna(subset=[timestamp_col, "ch1"]).copy()
rows_after_missing = len(df)
df = df.drop_duplicates(subset=[timestamp_col]).copy()
rows_after_duplicates = len(df)
df = df.sort_values(timestamp_col).reset_index(drop=True)

validation_report = pd.DataFrame({
    "metric": ["raw_rows", "removed_missing", "removed_duplicate_timestamps", "final_rows"],
    "value": [rows_before, rows_before - rows_after_missing, rows_after_missing - rows_after_duplicates, rows_after_duplicates],
})
display(validation_report)
display(missing_before.to_frame("missing_before_cleaning"))

## 5. Samplingrate prüfen

Die Pipeline erkennt Sekunden vs. Millisekunden, erzeugt `datetime`, berechnet Sample-Abstände und schätzt die Samplingrate.

In [ ]:
df, sampling_report = _check_sampling_rate(df, timestamp_col)
fs = sampling_report["estimated_fs_median_hz"]
display(pd.DataFrame([sampling_report]).drop(columns=["warnings"]))
if sampling_report["warnings"]:
    print("Warnungen:")
    for message in sampling_report["warnings"]:
        print("-", message)
else:
    print("Keine auffälligen Sampling-Warnungen.")

plt.figure()
intervals = df["sample_interval"].dropna()
plt.hist(intervals, bins=80, color="#4C78A8", alpha=0.85)
plt.title("Verteilung der Sample-Abstände")
plt.xlabel("Sample-Abstand [s]")
plt.ylabel("Anzahl")
savefig("05_sample_interval_distribution.png")

## 6. Rohsignal visualisieren

Gesamtes Rohsignal und Detailansicht der ersten 20 Sekunden.

In [ ]:
plt.figure()
plt.plot(df["datetime"], df["ch1"], linewidth=0.6)
plt.title("Rohsignal über Zeit")
plt.xlabel("Zeit")
plt.ylabel("ch1")
savefig("06_raw_signal.png")

zoom_df = df[df["time_seconds"] <= DETAIL_SECONDS].copy()
plt.figure()
plt.plot(zoom_df["datetime"], zoom_df["ch1"], linewidth=0.8)
plt.title("Rohsignal Detailansicht: erste konfigurierte Sekunden")
plt.xlabel("Zeit")
plt.ylabel("ch1")
savefig("06_raw_signal_first_20s.png")

## 7. Zentrierung und Filterung visualisieren

DC-Offset wird entfernt, danach folgen Butterworth-Bandpass 1-40 Hz und 50-Hz-Notch gegen Netzbrummen.

In [ ]:
df["ch1_raw"] = df["ch1"]
dc_offset = float(df["ch1_raw"].mean())
df["ch1_centered"] = df["ch1_raw"] - dc_offset

sos = butter(N=BANDPASS_ORDER, Wn=[BANDPASS_LOW_HZ, BANDPASS_HIGH_HZ], btype="bandpass", fs=fs, output="sos")
df["ch1_bandpassed"] = sosfiltfilt(sos, df["ch1_centered"].to_numpy(dtype=float))
if NOTCH_FREQ_HZ < fs / 2.0:
    b, a = iirnotch(w0=NOTCH_FREQ_HZ, Q=NOTCH_QUALITY_FACTOR, fs=fs)
    df["ch1_filtered"] = filtfilt(b, a, df["ch1_bandpassed"].to_numpy(dtype=float))
else:
    df["ch1_filtered"] = df["ch1_bandpassed"]

print(f"Entfernter DC-Offset: {dc_offset:.6f}")

plt.figure()
plt.plot(df["datetime"], df["ch1_raw"], linewidth=0.5, alpha=0.45, label="roh")
plt.plot(df["datetime"], df["ch1_centered"], linewidth=0.6, alpha=0.8, label="zentriert")
plt.title("Rohsignal vs. zentriertes Signal")
plt.xlabel("Zeit")
plt.ylabel("Amplitude")
plt.legend()
savefig("07_raw_vs_centered.png")

plt.figure()
plt.plot(df["datetime"], df["ch1_filtered"], linewidth=0.7, color="#F58518")
plt.title("Gefiltertes Signal 1-40 Hz + Notch")
plt.xlabel("Zeit")
plt.ylabel("Amplitude")
savefig("07_filtered_signal.png")

## 8. PSD vor/nach Filterung vergleichen

Welch-PSD zeigt, wie Bandpass und Notch das Spektrum verändern.

In [ ]:
freqs_before, psd_before = welch(df["ch1_centered"], fs=fs, nperseg=min(PSD_FULL_SIGNAL_NPERSEG_MAX, len(df)), detrend="constant", scaling="density")
freqs_after, psd_after = welch(df["ch1_filtered"], fs=fs, nperseg=min(PSD_FULL_SIGNAL_NPERSEG_MAX, len(df)), detrend="constant", scaling="density")

plt.figure(figsize=(12, 5))
plt.semilogy(freqs_before, psd_before, label="vor Filterung", alpha=0.7)
plt.semilogy(freqs_after, psd_after, label="nach Filterung", alpha=0.9)
plt.xlim(0, min(80, fs / 2))
plt.title("PSD vor/nach Filterung")
plt.xlabel("Frequenz [Hz]")
plt.ylabel("PSD")
plt.legend()
savefig("08_psd_before_after_filtering.png")

## 9. Fensterbildung darstellen

Das gefilterte Signal wird in 10-Sekunden-Fenster mit 50 % Overlap segmentiert. Jedes Fenster ist eine spätere Beobachtung.

In [ ]:
windows = _create_windows(df, fs, window_seconds=WINDOW_SECONDS, overlap=WINDOW_OVERLAP)
window_table = pd.DataFrame([{k: v for k, v in win.items() if k != "signal_segment"} for win in windows])
display(window_table.head())
print("Anzahl Fenster:", len(windows))

plt.figure(figsize=(14, 2.8))
for _, row in window_table.head(20).iterrows():
    plt.hlines(1, row["start_time"], row["end_time"], linewidth=6, alpha=0.7)
plt.title("Erste 20 überlappende Fenster")
plt.xlabel("Zeit")
plt.yticks([])
savefig("09_windowing_overview.png")

## 10. Artefakt-Erkennung analysieren

Artefakte werden mit robusten Schwellenwerten aus Peak-to-Peak, Standardabweichung, Maximalamplitude und Energie markiert.

In [ ]:
artifact_df = pd.DataFrame([_compute_artifact_metrics(win) for win in windows])
thresholds = _artifact_thresholds(artifact_df)
artifact_df["artifact"] = (
    (artifact_df["p2p"] > thresholds["p2p_threshold"]) |
    (artifact_df["std"] > thresholds["std_threshold"]) |
    (artifact_df["max_abs"] > thresholds["max_abs_threshold"]) |
    (artifact_df["energy"] > thresholds["energy_threshold"])
)

display(pd.DataFrame([thresholds]))
display(artifact_df.head())
print("Artefaktfenster:", int(artifact_df["artifact"].sum()), "von", len(artifact_df))

plt.figure()
colors = np.where(artifact_df["artifact"], "#E45756", "#4C78A8")
plt.scatter(artifact_df["start_time"], artifact_df["p2p"], c=colors, s=18)
plt.axhline(thresholds["p2p_threshold"], color="#E45756", linestyle="--", label="P2P Schwelle")
plt.title("Artefaktübersicht über Fenster")
plt.xlabel("Zeit")
plt.ylabel("Peak-to-Peak")
plt.legend()
savefig("10_artifact_overview.png")

## 11. Saubere vs. Artefaktfenster vergleichen

Die Verteilungen der Artefaktmetriken machen sichtbar, wodurch Fenster markiert wurden.

In [ ]:
metrics = ["p2p", "std", "max_abs", "energy"]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, metric in zip(axes.ravel(), metrics):
    clean = artifact_df.loc[~artifact_df["artifact"], metric]
    art = artifact_df.loc[artifact_df["artifact"], metric]
    ax.hist(clean, bins=40, alpha=0.75, label="clean", color="#4C78A8")
    ax.hist(art, bins=40, alpha=0.75, label="artifact", color="#E45756")
    ax.set_title(metric)
    ax.legend()
savefig("11_clean_vs_artifact_metrics.png")

## 12. Bandpower berechnen und visualisieren

PSD und Bandpower werden nur für saubere Fenster auf dem gefilterten Signal berechnet.

In [ ]:
feature_rows = []
psd_examples = []
artifact_lookup = artifact_df.set_index("window_id")["artifact"].to_dict()

for win in windows:
    is_artifact = bool(artifact_lookup[win["window_id"]])
    row = {
        "window_id": int(win["window_id"]),
        "start_sample": int(win["start_sample"]),
        "end_sample": int(win["end_sample"]),
        "start_time": win["start_time"],
        "end_time": win["end_time"],
        "artifact": is_artifact,
    }
    if is_artifact:
        feature_rows.append(row)
        continue
    freqs, psd = welch(win["signal_segment"], fs=fs, nperseg=min(PSD_NPERSEG_MAX, len(win["signal_segment"])), detrend="constant", scaling="density")
    if len(psd_examples) < 3:
        psd_examples.append((win["window_id"], freqs, psd))
    absolute = {f"absolute_{name}": _bandpower(freqs, psd, band) for name, band in FREQUENCY_BANDS.items()}
    total_power = _bandpower(freqs, psd, TOTAL_POWER_BAND)
    relative = {f"relative_{name}": absolute[f"absolute_{name}"] / (total_power + EPS) for name in FREQUENCY_BANDS}
    theta = absolute["absolute_theta"]
    alpha = absolute["absolute_alpha"]
    beta = absolute["absolute_beta"]
    signal = win["signal_segment"]
    row.update({
        **absolute,
        "total_power_1_40hz": total_power,
        **relative,
        "theta_alpha_ratio": theta / (alpha + EPS),
        "alpha_theta_ratio": alpha / (theta + EPS),
        "beta_alpha_ratio": beta / (alpha + EPS),
        "signal_mean": float(np.mean(signal)),
        "signal_std": float(np.std(signal)),
        "signal_variance": float(np.var(signal)),
        "peak_to_peak": float(np.ptp(signal)),
        "max_abs": float(np.max(np.abs(signal))),
        "spectral_entropy": _spectral_entropy(psd),
    })
    feature_rows.append(row)

features_df = pd.DataFrame(feature_rows).sort_values("window_id").reset_index(drop=True)
clean_features_df = features_df[features_df["artifact"] == False].copy()
display(features_df.head())
print("Saubere Feature-Fenster:", len(clean_features_df))

if psd_examples:
    plt.figure(figsize=(12, 5))
    win_id, freqs, psd = psd_examples[0]
    plt.semilogy(freqs, psd)
    plt.xlim(0, 45)
    plt.title(f"Beispiel-PSD Fenster {win_id}")
    plt.xlabel("Frequenz [Hz]")
    plt.ylabel("PSD")
    savefig("12_example_psd.png")

plt.figure()
for col in ["relative_delta", "relative_theta", "relative_alpha", "relative_beta", "relative_gamma"]:
    plt.plot(clean_features_df["start_time"], clean_features_df[col], label=col, linewidth=0.9)
plt.title("Relative Bandpower über Zeit")
plt.xlabel("Zeit")
plt.ylabel("Relative Bandpower")
plt.legend()
savefig("12_relative_bandpower.png")

## 13. Relative Features und Ratios analysieren

Theta/Alpha, Beta/Alpha und relative Bandpower sind zentrale EEG-Marker. Gamma wird vorsichtig interpretiert, weil Muskelartefakte bei Single-Channel/In-Ear-EEG stark einwirken können.

In [ ]:
ratio_cols = ["theta_alpha_ratio", "beta_alpha_ratio", "alpha_theta_ratio"]
plt.figure()
for col in ratio_cols:
    plt.plot(clean_features_df["start_time"], clean_features_df[col], label=col, linewidth=0.9)
plt.title("EEG Ratios über Zeit")
plt.xlabel("Zeit")
plt.ylabel("Ratio")
plt.legend()
savefig("13_ratios_over_time.png")

display(clean_features_df[["relative_theta", "relative_alpha", "relative_beta", "relative_gamma", *ratio_cols]].describe().T)

## 14. Cognitive Load Proxy Score visualisieren

Der Score wird aus baseline-normalisierten EEG-Features berechnet. Er ist kein validierter Overload-Labelersatz.

In [ ]:
clean_scored_df, baseline_report = _baseline_normalize(clean_features_df, baseline_minutes=BASELINE_MINUTES)
clean_scored_df["Cognitive_Load_Proxy_Score"] = _compute_proxy_score(clean_scored_df)
clean_scored_df, score_thresholds = _assign_load_states(clean_scored_df)
clean_scored_df["Cognitive_Load_Proxy_Score_Smoothed"] = clean_scored_df["Cognitive_Load_Proxy_Score"].rolling(ROLLING_WINDOWS, min_periods=1, center=True).mean()

score_merge_cols = [
    "window_id", "baseline_window",
    "theta_alpha_ratio_z", "beta_alpha_ratio_z", "relative_theta_z",
    "relative_alpha_z", "spectral_entropy_z",
    "Cognitive_Load_Proxy_Score", "Cognitive_Load_Proxy_Score_Smoothed",
    "cognitive_load_state",
]
features_df = features_df.merge(clean_scored_df[score_merge_cols], on="window_id", how="left")
features_df["artifact"] = features_df["artifact"].fillna(False).astype(bool)
features_df.loc[features_df["artifact"], "cognitive_load_state"] = "artifact"
features_df, deviation_report = _add_eeg_deviation_index(features_df)

display(pd.DataFrame([baseline_report]))
display(pd.DataFrame([score_thresholds]))

plt.figure()
plt.plot(clean_scored_df["start_time"], clean_scored_df["Cognitive_Load_Proxy_Score"], label="Proxy", linewidth=0.8)
plt.plot(clean_scored_df["start_time"], clean_scored_df["Cognitive_Load_Proxy_Score_Smoothed"], label="geglättet", linewidth=1.4)
plt.axhline(score_thresholds["elevated_threshold"], color="#F58518", linestyle="--", label="erhöht")
plt.axhline(score_thresholds["strongly_elevated_threshold"], color="#E45756", linestyle="--", label="stark erhöht")
plt.title("Cognitive Load Proxy Score")
plt.xlabel("Zeit")
plt.ylabel("baseline-relativer EEG-Score")
plt.legend()
savefig("14_cognitive_load_proxy_score.png")

## 15. Hochlastphasen anzeigen

Hier werden nur EEG-basierte, baseline-relative Hochlastphasen angezeigt. Ohne externen Score sind das keine bewiesenen Overload-Zustände.

In [ ]:
state_colors = {"baseline-nah": "#4C78A8", "erhöht": "#F58518", "stark erhöht": "#E45756", "artifact": "#9D9D9D"}
plt.figure()
for state, color in state_colors.items():
    state_df = features_df[features_df["cognitive_load_state"] == state]
    if state_df.empty:
        continue
    plt.scatter(state_df["start_time"], state_df["Cognitive_Load_Proxy_Score"], label=state, color=color, s=22)
plt.title("Proxy-Zustände über Zeit")
plt.xlabel("Zeit")
plt.ylabel("Cognitive Load Proxy Score")
plt.legend()
savefig("15_proxy_states.png")

display(features_df["cognitive_load_state"].value_counts(dropna=False).to_frame("windows"))

## 16. Feature-Korrelationen darstellen

Korrelationsmuster helfen, redundante Features und Score-Treiber zu erkennen.

In [ ]:
corr_cols = [
    "relative_delta", "relative_theta", "relative_alpha", "relative_beta", "relative_gamma",
    "theta_alpha_ratio", "beta_alpha_ratio", "alpha_theta_ratio", "spectral_entropy",
    "Cognitive_Load_Proxy_Score", "EEG_Deviation_Index",
]
corr_data = features_df.loc[features_df["artifact"] == False, [col for col in corr_cols if col in features_df.columns]].dropna()
corr = corr_data.corr()

plt.figure(figsize=(10, 8))
im = plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(im, label="Korrelation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Feature-Korrelationen")
savefig("16_feature_correlation_heatmap.png")
display(corr.round(3))

## 17. Falls NASA-TLX vorhanden: externe Validierung

NASA-TLX oder ein anderer externer Workload-Score wird nicht als EEG-Merkmal verwendet. Er dient nur als Referenz/Ground Truth.

In [ ]:
if external_score_available:
    features_df, external_report = _attach_and_validate_external_score(
        signal_df=df,
        features_df=features_df,
        windows=windows,
        score_thresholds=score_thresholds,
        external_score_column=EXTERNAL_SCORE_COLUMN,
        external_score_type=EXTERNAL_SCORE_TYPE,
        high_workload_threshold=HIGH_WORKLOAD_THRESHOLD,
        plots_dir=PLOT_DIR,
    )
    display(pd.DataFrame([external_report["continuous_metrics"]]))
    display(pd.DataFrame([external_report["binary_metrics"]]).drop(columns=["confusion_matrix"], errors="ignore"))
    display(pd.DataFrame([external_report["binary_metrics"]["confusion_matrix"]]))

    valid_ext = features_df[(features_df["artifact"] == False) & features_df[EXTERNAL_SCORE_COLUMN].notna()].copy()
    plt.figure(figsize=(7, 6))
    colors = np.where(valid_ext["high_workload"], "#E45756", "#4C78A8")
    plt.scatter(valid_ext[EXTERNAL_SCORE_COLUMN], valid_ext["Cognitive_Load_Proxy_Score"], c=colors, alpha=0.75)
    plt.title("EEG Score vs. NASA-TLX/externer Score")
    plt.xlabel(EXTERNAL_SCORE_COLUMN)
    plt.ylabel("Cognitive_Load_Proxy_Score")
    savefig("17_eeg_score_vs_external_score.png")

    plt.figure(figsize=(8, 5))
    valid_ext.boxplot(column="Cognitive_Load_Proxy_Score", by="high_workload", grid=True)
    plt.suptitle("")
    plt.title("Low/High Workload Vergleich")
    plt.xlabel("high_workload nach externem Score")
    plt.ylabel("Cognitive_Load_Proxy_Score")
    savefig("17_low_high_workload_boxplot.png")
else:
    external_report = {"available": False, "message": "Kein externer Workload-Score vorhanden."}
    print(external_report["message"])

## 18. Automatische Interpretation anzeigen

Die Interpretation bleibt bewusst vorsichtig und trennt EEG-Proxy, Signalqualität und externe Validierung.

In [ ]:
artifact_percent = 100 * artifact_df["artifact"].mean()
mean_score = clean_scored_df["Cognitive_Load_Proxy_Score"].mean()
state_counts = features_df["cognitive_load_state"].value_counts(normalize=True).mul(100).round(2).to_dict()

interpretation = []
interpretation.append(f"Artefaktanteil: {artifact_percent:.2f} %. Niedriger ist besser fuer Feature-Stabilitaet.")
interpretation.append(f"Mittlerer Cognitive Load Proxy Score: {mean_score:.3f}. Das ist ein baseline-relativer EEG-Index.")
interpretation.append(f"Verteilung der Proxy-Zustaende: {state_counts}.")
if external_report.get("available"):
    cm = external_report["continuous_metrics"]
    interpretation.append(f"Externe Validierung vorhanden: Pearson r={cm['pearson_r']}, Spearman rho={cm['spearman_rho']}.")
else:
    interpretation.append("Keine externe Validierung vorhanden: Es wird kein Cognitive Overload behauptet.")
interpretation.append("Gamma 30-40 Hz ist enthalten, sollte bei Single-Channel/In-Ear-EEG aber wegen EMG-Artefakten vorsichtig interpretiert werden.")

for line in interpretation:
    print("-", line)

## 19. Exportübersicht anzeigen

Notebook-Zwischenergebnisse werden separat exportiert. Die Hauptpipeline kann ueber `01_ordered_eeg_pipeline.py` oder den Wrapper `00_run_pipeline.py` ausgefuehrt werden.

In [ ]:
processed_path = EXPORT_DIR / f"{csv_path.stem}_notebook_processed_signal.csv"
features_path = EXPORT_DIR / f"{csv_path.stem}_notebook_features.csv"
artifact_path = EXPORT_DIR / f"{csv_path.stem}_notebook_artifact_windows.csv"

df.to_csv(processed_path, index=False)
features_df.to_csv(features_path, index=False)
artifact_df.to_csv(artifact_path, index=False)

exports = pd.DataFrame([
    {"artifact": "processed_signal", "path": processed_path},
    {"artifact": "features", "path": features_path},
    {"artifact": "artifact_windows", "path": artifact_path},
    {"artifact": "plot_directory", "path": PLOT_DIR},
])
display(exports)